# Глава 7. Тонкая настройка по инструкциям

In [1]:
from importlib.metadata import version

pkgs = [
    "numpy",       # Зависимость для PyTorch и TensorFlow
    "matplotlib",  # Библиотека для построения графиков
    "tiktoken",    # Токенизатор
    "torch",       # Библиотека для глубокого обучения
    "tqdm",        # Прогресс-бар
    "tensorflow",  # Для предобученных весов OpenAI
]
for p in pkgs:
    print(f"{p} версия: {version(p)}")

numpy версия: 2.4.6
matplotlib версия: 3.10.9
tiktoken версия: 0.13.0
torch версия: 2.12.0
tqdm версия: 4.67.3
tensorflow версия: 2.21.0


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/01.webp" width=800px>

&nbsp;
## 7.1. Введение в тонкую настройку по инструкциям

- Предобучение LLM включает в себя процедуру обучения, в ходе которой она учится генерировать по одному слову за раз
- Следовательно, предобученная LLM хорошо справляется с дополнением текста, но плохо следует инструкциям
- В этой главе мы научим LLM лучше следовать инструкциям

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/02.webp" width=800px>

&nbsp;
## 7.2. Подготовка набора данных для контролируемой тонкой настройки по инструкции

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/03.webp" width=800px>

- Мы будем работать с набором инструктивных данных

In [2]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


# Изначально использовался код ниже.
# Однако urllib использует старые настройки протокола,
# что может вызывать проблемы у некоторых читателей, использующих VPN.
# Версия с `requests` более надёжна в этом отношении.

"""
import urllib

def download_and_load_file(file_path, url):

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data
"""


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Количество записей:", len(data))

Количество записей: 1100


- Каждый элемент в списке `data`, который мы загрузили из JSON-файла выше, представляет собой словарь следующего вида

In [3]:
print("Пример записи:\n", data[50])

Пример записи:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


- Обратите внимание, что поле `'input'` может быть пустым:

In [4]:
print("Ещё один пример записи:\n", data[999])

Ещё один пример записи:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


- Тонкую настройку по инструкции часто называют «инструктивной тонкой настройкой с учителем», поскольку она включает обучение модели на наборе данных, где пары «вход-выход» явно заданы
- Существуют различные способы форматирования записей в качестве входных данных для LLM; на рисунке ниже показаны два примера форматов, которые использовались для обучения LLM Alpaca (https://crfm.stanford.edu/2023/03/13/alpaca.html) и Phi-3 (https://arxiv.org/abs/2404.14219) соответственно

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/04.webp?2" width=800px>

 ---

 ## Упражнение 7.1. Изменение стилей подсказок

Предположим, у нас есть следующая запись данных:

```json
{
  "instruction": "Identify the correct spelling of the following word.",
  "input": "Ocassion",
  "output": "The correct spelling is 'Occasion.'"
}
```

Мы форматировали её в соответствии с шаблоном промптов в стиле Alpaca:

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Occassion

### Response:
The correct spelling is 'Occasion.'
```

В этом упражнении мы теперь используем шаблон промптов Phi-3, который форматирует запись данных следующим образом:

```
<user>
Identify the correct spelling of the following word: 'Occasion'

<assistant>
The correct spelling is 'Occasion'.
```

Обратите внимание, что этот шаблон промптов существенно короче, что снижает требования к времени выполнения и аппаратному обеспечению для тонкой настройки LLM и генерации текста, поскольку входные промпты короче.
Чтобы внести это изменение, мы обновляем функцию `format_input` следующим образом:

In [5]:
def format_input(entry):
    instruction_text = (
        f"<|user|>\n{entry['instruction']}"
    )

    input_text = f"\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [6]:
sample_data = [
    {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}, 
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}
]

print(format_input(sample_data[0]))
print()
print(format_input(sample_data[1]))

<|user|>
Identify the correct spelling of the following word.
Ocassion

<|user|>
What is an antonym of 'complicated'?


Далее мы также обновим класс `InstructionDataset`, чтобы он использовал шаблон промпта `<|assistant|>` для ответа:

Давайте убедимся, что это работает как задумано, применив её к двум входным образцам — одному с содержимым в поле `'input'` и одному без:

In [7]:
import tiktoken
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Предварительно токенизируем тексты
        self.encoded_texts = []
        for entry in data:

            ###################################################################
            # НОВОЕ: Используем `format_input_phi` и изменяем шаблон текста ответа
            instruction_plus_input = format_input(entry)
            response_text = f"\n<|assistant|>:\n{entry['output']}"
            ###################################################################
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


tokenizer = tiktoken.get_encoding("gpt2")

Наконец, мы также должны обновить способ извлечения сгенерированного ответа при сборе ответов тестового набора:

```python
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)
    tokenizer=tokenizer

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    # Новое: Заменяем ###Response на <|assistant|>
    response_text = generated_text[len(input_text):].replace("<|assistant|>:", "").strip()

    test_data[i]["model_response"] = response_text
```

Для вашего удобства решение упражнения реализовано в скрипте [exercise_experiments.py](exercise_experiments.py):

```bash
python exercise_experiments.py --exercise_solution phi3_prompt
```

Вывод:

```
matplotlib версия: 3.7.1
tiktoken версия: 0.7.0
torch версия: 2.3.0+cu121
tqdm версия: 4.66.4
tensorflow версия: 2.15.0
--------------------------------------------------
Длина обучающего набора: 935
Длина валидационного набора: 55
Длина тестового набора: 110
--------------------------------------------------
Устройство: cuda
--------------------------------------------------
...
Загружена модель: gpt2-medium (355M)
--------------------------------------------------
Начальные потери
   Потери на обучении: 3.71630220413208
   Потери на валидации: 3.6440994262695314
Эп 1 (Шаг 000000): Потери на обучении 2.633, Потери на валидации 2.622
...
Эп 2 (Шаг 000230): Потери на обучении 0.424, Потери на валидации 0.928
<|user|> Convert the active sentence to passive: 'The chef cooks the meal every day.' <|assistant|>: The meal is prepared every day by the chef....
Обучение завершено за 1.50 минут.
График сохранён как loss-plot-phi3-prompt.pdf
--------------------------------------------------
Генерация ответов
100% 110/110 [00:11<00:00,  9.27it/s]
Ответы сохранены как instruction-data-with-response-phi3-prompt.json
Модель сохранена как gpt2-medium355M-sft-phi3-prompt.pth
```

Для сравнения вы можете запустить оригинальный код тонкой настройки из главы 7 с помощью `python exercise_experiments.py --exercise_solution baseline`. 

Обратите внимание, что на GPU Nvidia L4 приведённый выше код с использованием шаблона промптов Phi-3 выполняется за 1,5 минуты. Для сравнения, шаблон в стиле Alpaca выполняется за 1,80 минуты. Таким образом, шаблон Phi-3 примерно на 17% быстрее, поскольку он приводит к более коротким входным данным модели. 

Давайте посмотрим на некоторые ответы, чтобы убедиться, что они отформатированы правильно:

```json
    {
        "instruction": "Rewrite the sentence using a simile.",
        "input": "The car is very fast.",
        "output": "The car is as fast as lightning.",
        "model_response": "The car is as fast as a cheetah."
    },
    {
        "instruction": "What type of cloud is typically associated with thunderstorms?",
        "input": "",
        "output": "The type of cloud typically associated with thunderstorms is cumulonimbus.",
        "model_response": "The type of cloud associated with thunderstorms is a cumulus cloud."
    },
    {
        "instruction": "Name the author of 'Pride and Prejudice'.",
        "input": "",
        "output": "Jane Austen.",
        "model_response": "The author of 'Pride and Prejudice' is Jane Austen."
    },
```

Мы можем оценить производительность с помощью метода Ollama Llama 3, который для вашего удобства также реализован в скрипте `python exercise_experiments.py` и который можно запустить следующим образом:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-phi3-prompt.json
```

Вывод:

```
Ollama запущен: True
Подсчёт оценок: 100%|████████████████████████| 110/110 [01:08<00:00,  1.60it/s]
Количество оценок: 110 из 110
Средняя оценка: 48.87
```

Оценка близка к 50, что находится в том же диапазоне, что и оценка, которую мы ранее получили с промптами в стиле Alpaca.

Нет никакого неотъемлемого преимущества или обоснования того, почему стиль промптов Phi должен быть лучше, но он может быть более кратким и эффективным, за исключением оговорки, упомянутой в разделе *Совет* ниже.

#### Совет: Учёт специальных токенов

- Обратите внимание, что шаблон промптов Phi-3 содержит специальные токены, такие как `<|user|>` и `<|assistant|>`, что может быть неоптимальным для токенизатора GPT-2
- Хотя токенизатор GPT-2 распознаёт `<|endoftext|>` как специальный токен (кодируемый в идентификатор токена 50256), он неэффективно обрабатывает другие специальные токены, такие как вышеупомянутые
- Например, `<|user|>` кодируется в 5 отдельных идентификаторов токенов (27, 91, 7220, 91, 29), что очень неэффективно
- Мы могли бы добавить `<|user|>` как новый специальный токен в `tiktoken` через аргумент `allowed_special`, но имейте в виду, что словарь GPT-2 не сможет обработать его без дополнительных модификаций
- Если вам интересно, как токенизатор и LLM могут быть расширены для обработки специальных токенов, пожалуйста, ознакомьтесь с дополнительными материалами [extend-tiktoken.ipynb](../../ch05/09_extending-tokenizers/extend-tiktoken.ipynb) (обратите внимание, что это не обязательно здесь, но представляет собой интересное/бонусное соображение для любознательных читателей)
- Кроме того, мы можем предположить, что модели, которые поддерживают эти специальные токены шаблона промптов через свой словарь, могут работать более эффективно и лучше в целом

---

- Мы используем форматирование промптов в стиле Alpaca, которое было исходным шаблоном промптов для инструктивной тонкой настройки
- Ниже мы форматируем входные данные, которые будем передавать в LLM

In [8]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

- Отформатированный ответ с полем ввода выглядит так, как показано ниже

In [9]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


- Ниже представлен отформатированный ответ без поля ввода

In [10]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


- Наконец, прежде чем мы подготовим загрузчики данных PyTorch в следующем разделе, мы разделим набор данных на обучающую, валидационную и тестовую выборки

In [11]:
train_portion = int(len(data) * 0.85)  # 85% на обучение
test_portion = int(len(data) * 0.1)    # 10% на тестирование
val_portion = len(data) - train_portion - test_portion  # Оставшиеся 5% на валидацию

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [12]:
print("Длина обучающего набора:", len(train_data))
print("Длина валидационного набора:", len(val_data))
print("Длина тестового набора:", len(test_data))

Длина обучающего набора: 935
Длина валидационного набора: 55
Длина тестового набора: 110


&nbsp;
## 7.3 Организация данных в обучающие пакеты

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/05.webp?1" width=800px>

- Мы решаем задачу пакетирования этого набора данных в несколько этапов, как показано на рисунке ниже

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/06.webp?1" width=800px>

- Сначала мы реализуем класс `InstructionDataset`, который предварительно токенизирует все входные данные в наборе данных, аналогично классу `SpamDataset`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/07.webp?1" width=800px>

In [13]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Предварительно токенизируем тексты
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- Мы хотим объединить несколько обучающих примеров в пакет, чтобы ускорить обучение; для этого необходимо дополнить все входные данные до одинаковой длины
- Также, как и в предыдущей главе, мы используем токен `<|endoftext|>` в качестве токена заполнения

In [14]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


- Ранее мы дополнили все примеры в наборе данных до одинаковой длины
  - Здесь мы используем более изощрённый подход и разрабатываем пользовательскую функцию «collate», которую можно передать в загрузчик данных
  - Эта пользовательская функция collate дополняет обучающие примеры в каждом пакете так, чтобы они имели одинаковую длину (но разные пакеты могут иметь разную длину)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/08.webp?1" width=800px>

In [15]:
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Находим самую длинную последовательность в пакете
    # и увеличиваем максимальную длину на +1, что добавит один дополнительный
    # токен заполнения ниже
    batch_max_length = max(len(item)+1 for item in batch)

    # Дополняем и подготавливаем входы
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        # Добавляем токен <|endoftext|>
        new_item += [pad_token_id]
        # Дополняем последовательности до batch_max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # Через padded[:-1] мы удаляем дополнительный токен заполнения,
        # который был добавлен из-за +1 в batch_max_length
        # (дополнительный токен заполнения будет важен в последующем коде)
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    # Преобразуем список входов в тензор и переносим на целевое устройство
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [16]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/09.webp?1" width=800px>

- Выше мы вернули только входные данные для LLM; однако для обучения LLM нам также нужны целевые значения
- Как и при предобучении LLM, цели представляют собой входные данные, сдвинутые на 1 позицию вправо, чтобы LLM училась предсказывать следующий токен

In [17]:
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Находим самую длинную последовательность в пакете
    batch_max_length = max(len(item)+1 for item in batch)

    # Дополняем и подготавливаем входы и цели
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Добавляем токен <|endoftext|>
        new_item += [pad_token_id]
        # Дополняем последовательности до max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Обрезаем последний токен для входов
        targets = torch.tensor(padded[1:])  # Сдвигаем на +1 вправо для целей
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Преобразуем списки входов и целей в тензоры и переносим на целевое устройство
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [18]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


- Далее мы вводим значение `ignore_index`, чтобы заменить все идентификаторы токенов заполнения новым значением; цель этого `ignore_index` заключается в том, что мы можем игнорировать значения заполнения в функции потерь (подробнее об этом позже)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/11.webp?1" width=800px>

- Конкретно это означает, что мы заменяем идентификаторы токенов, соответствующие `50256`, на `-100`, как показано ниже

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/12.webp?2" width=800px>

- (Кроме того, мы также вводим `allowed_max_length` на случай, если захотим ограничить длину образцов; это будет полезно, если мы планируем работать с собственными наборами данных, длина которых превышает размер контекста в 1024 токена, поддерживаемый моделью GPT-2)

In [19]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Находим самую длинную последовательность в пакете
    batch_max_length = max(len(item)+1 for item in batch)

    # Дополняем и подготавливаем входы и цели
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Добавляем токен <|endoftext|>
        new_item += [pad_token_id]
        # Дополняем последовательности до max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Обрезаем последний токен для входов
        targets = torch.tensor(padded[1:])  # Сдвигаем на +1 вправо для целей

        # Новое: заменяем все, кроме первого, токены заполнения в целях на ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze() # .squeeze() превращает многомерный массив в обычный
        if indices.numel() > 1: # .numel() возвращает количество элементов в тензоре
            targets[indices[1:]] = ignore_index

        # Новое: опционально обрезаем до максимальной длины последовательности
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Преобразуем списки входов и целей в тензоры и переносим на целевое устройство
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [20]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


- Давайте посмотрим, чего позволяет добиться эта замена на -100
- Для наглядности предположим, что у нас есть небольшая задача классификации с 2 метками классов, 0 и 1
- Если у нас есть следующие значения логитов (выходы последнего слоя модели), мы вычисляем следующую функцию потерь

In [21]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1-й обучающий пример
     [-0.5, 1.5]]  # 2-й обучающий пример
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


- Теперь добавление ещё одного обучающего примера, как и ожидалось, повлияет на функцию потерь

In [22]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # Новый 3-й обучающий пример
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


- Давайте посмотрим, что произойдёт, если мы заменим метку класса одного из примеров на -100

In [23]:
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


- Как мы видим, результирующая функция потерь на этих 3 обучающих примерах совпадает с функцией потерь, которую мы вычислили на 2 обучающих примерах, что означает, что функция кросс-энтропийных потерь проигнорировала обучающий пример с меткой -100
- По умолчанию в PyTorch используется настройка `cross_entropy(..., ignore_index=-100)`, позволяющая игнорировать примеры, соответствующие метке -100
- Используя этот `ignore_index` со значением -100, мы можем игнорировать дополнительные токены конца текста (заполнения) в пакетах, которые мы использовали для дополнения обучающих примеров до одинаковой длины
- Однако мы не хотим игнорировать первый экземпляр токена конца текста (заполнения) (50256), потому что он может помочь сигнализировать LLM о завершении ответа

- На практике также распространено маскирование целевых идентификаторов токенов, соответствующих инструкции, как показано на рисунке ниже

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/13.webp" width=800px>

 ---
&nbsp;
## Упражнение 7.2: Маскировка инструкций и входных данных

Чтобы замаскировать инструкции, как показано на следующем рисунке, нам необходимо внести небольшие изменения в класс `InstructionDataset` и функцию `custom_collate_fn`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/mask-instructions.webp" width=800px>

In [24]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

Мы можем модифицировать класс `InstructionDataset`, чтобы он собирал длины инструкций, которые мы будем использовать в функции collate для определения позиций содержимого инструкций в целях при написании функции collate, следующим образом:

In [25]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        ##########################################################################################
        # Новое: отдельный список для длин инструкций
        self.instruction_lengths = []
        ##########################################################################################
        
        self.encoded_texts = []
        
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

            ##########################################################################################
            # Новое: собираем длины инструкций
            instruction_length = len(tokenizer.encode(instruction_plus_input))
            self.instruction_lengths.append(instruction_length)
            ##########################################################################################
            
    def __getitem__(self, index):
        # Новое: возвращаем и длины инструкций, и тексты по отдельности
        return self.instruction_lengths[index], self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [26]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

Далее мы обновим `custom_collate_fn`, где каждый `batch` теперь является кортежем, содержащим `(instruction_length, item)`, а не просто `item`, из-за изменений в классе `InstructionDataset`. Кроме того, теперь мы маскируем соответствующие токены инструкций в списке целевых идентификаторов

In [27]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Находим самую длинную последовательность в пакете
    batch_max_length = max(len(item)+1 for instruction_length, item in batch)   # Новое: пакет теперь является кортежем

    # Дополняем и подготавливаем входы и цели
    inputs_lst, targets_lst = [], []

    for instruction_length, item in batch:  # Новое: пакет теперь является кортежем
        new_item = item.copy()
        # Добавляем токен <|endoftext|>
        new_item += [pad_token_id]
        # Дополняем последовательности до max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # Обрезаем последний токен для входов
        targets = torch.tensor(padded[1:])  # Сдвигаем на +1 вправо для целей

        # Заменяем все, кроме первого, токены заполнения в целях на ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        ##########################################################################################
        # Новое: маскируем все токены входа и инструкции в целях
        targets[:instruction_length-1] = -100
        ##########################################################################################
        
        # Опционально обрезаем до максимальной длины последовательности
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Преобразуем списки входов и целей в тензоры и переносим на целевое устройство
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

Давайте опробуем это на нескольких образцах данных ниже:

In [28]:
sample_data = [
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."},
    {'instruction': 'Sort the following list in alphabetical order.', 'input': 'Zebra, Elephant, Crocodile', 'output': 'Crocodile, Elephant, Zebra'},
    {'instruction': 'Arrange the given numbers in descending order.', 'input': '5, 12, 8, 3, 15', 'output': '15, 12, 8, 5, 3.'}
]

In [29]:
from torch.utils.data import DataLoader

train_dataset = InstructionDataset(sample_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=len(sample_data),
    collate_fn=custom_collate_fn,
    num_workers=0
)

In [30]:
print("Загрузчик обучающих данных:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Загрузчик обучающих данных:
torch.Size([3, 64]) torch.Size([3, 64])


In [31]:
print("Входы:\n", inputs[1])
print("\n\nЦели:\n", targets[1])

Входы:
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 42758,   262,  1708,  1351,   287, 24830,
          605,  1502,    13,   198,   198, 21017, 23412,    25,   198,    57,
        37052,    11, 42651,    11,  9325, 19815,   576,   198,   198, 21017,
        18261,    25,   198,    34, 12204,   375,   576,    11, 42651,    11,
         1168, 37052, 50256, 50256])


Цели:
 tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,   198,   198, 21017, 18261,
           25,   198,    34, 12204,   375,   576,    11, 42651,    11,  1168,
        37

Как мы видим на основе тензора `targets`, и токены инструкций, и токены заполнения теперь замаскированы с помощью токенов-заполнителей -100.
Декодируем входные данные, чтобы убедиться, что они выглядят правильно:

In [32]:
print(tokenizer.decode(list(inputs[1])))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Sort the following list in alphabetical order.

### Input:
Zebra, Elephant, Crocodile

### Response:
Crocodile, Elephant, Zebra<|endoftext|><|endoftext|>


Далее давайте декодируем незамаскированные идентификаторы целевых токенов:

In [33]:
non_masked_targets = targets[1][targets[1] != -100]

print(tokenizer.decode(list(non_masked_targets)))



### Response:
Crocodile, Elephant, Zebra<|endoftext|>


Как показано выше, незамаскированные целевые токены исключают поля `"Instruction"` и `"Input"`, как и задумано. Теперь мы можем запустить модифицированный код, чтобы увидеть, насколько хорошо LLM работает при тонкой настройке с использованием этой стратегии маскирования.

Для вашего удобства можно использовать код `exercise_experiments.py` для сравнительного запуска следующим образом:

```bash
python exercise_experiments.py --exercise_solution mask_instructions
```

Вывод:

```
matplotlib версия: 3.7.1
tiktoken версия: 0.7.0
torch версия: 2.3.0+cu121
tqdm версия: 4.66.4
tensorflow версия: 2.15.0
--------------------------------------------------
Длина обучающего набора: 935
Длина валидационного набора: 55
Длина тестового набора: 110
--------------------------------------------------
Устройство: cuda
--------------------------------------------------
...
Загружена модель: gpt2-medium (355M)
--------------------------------------------------
Начальные потери
   Потери на обучении: 2.280539035797119
   Потери на валидации: 2.262560224533081
Эп 1 (Шаг 000000): Потери на обучении 1.636, Потери на валидации 1.620
...
Эп 2 (Шаг 000230): Потери на обучении 0.143, Потери на валидации 0.727
...
Обучение завершено за 1.77 минут.
График сохранён как loss-plot-mask-instructions.pdf
--------------------------------------------------
Генерация ответов
100% 110/110 [02:10<00:00,  1.19s/it]
Ответы сохранены как instruction-data-with-response-mask-instructions.json
Модель сохранена как gpt2-medium355M-sft-mask-instructions.pth
```

Далее давайте оценим производительность полученной LLM:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-mask-instructions.json
```

```
Ollama запущен: True
Подсчёт оценок: 100%|██████████████████████████████████████████████████████████████████████████████████████| 110/110 [01:23<00:00,  1.31it/s]
Количество оценок: 110 из 110
Средняя оценка: 47.73
```

Как мы видим на основе оценок, маскирование инструкций действительно работает немного хуже, что согласуется с наблюдениями в статье "Instruction Tuning With Loss Over Instructions" (https://arxiv.org/abs/2405.14394).

---

&nbsp;
## 7.4 Создание загрузчиков данных для набора инструкций

- В этом разделе мы используем класс `InstructionDataset` и функцию `custom_collate_fn` для создания экземпляров загрузчиков обучающих, валидационных и тестовых данных

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/14.webp" width=800px>

- Ещё одна дополнительная деталь предыдущей функции `custom_collate_fn` заключается в том, что теперь мы перемещаем данные на целевое устройство (например, GPU) непосредственно в ней, а не в основном цикле обучения, что повышает эффективность, поскольку это может выполняться как фоновый процесс, когда мы используем `custom_collate_fn` как часть загрузчика данных
- Используя функцию `partial` из стандартной библиотеки Python `functools`, мы создаём новую функцию с предварительно заполненным аргументом `device` исходной функции

In [34]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Используйте PyTorch 2.9 или новее для стабильных результатов mps
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Устройство:", device)

Устройство: cpu


In [35]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

- Далее мы создаём экземпляры загрузчиков данных аналогично предыдущим главам, за исключением того, что теперь мы предоставляем собственную функцию collate для процесса пакетирования

In [36]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [37]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

- Давайте посмотрим, как выглядят размеры результирующих входных и целевых пакетов

In [38]:
print("Загрузчик обучающих данных:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Загрузчик обучающих данных:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([

- Как мы видим из вывода выше, все пакеты имеют размер пакета 8, но разную длину, как и ожидалось
- Давайте также дважды проверим, что входные данные содержат токены заполнения `<|endoftext|>`, соответствующие идентификатору токена 50256, выведя содержимое первого обучающего примера в пакете `inputs`

In [39]:
print(inputs[0])

tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,
          985,   576,    13,   198,   198, 21017, 23412,    25,   198,   464,
         5156,   318,   845, 13779,    13,   198,   198, 21017, 18261,    25,
          198,   464,  5156,   318,   355, 13779,   355,   257,  4936,    13,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256])


- Аналогично мы визуально перепроверим, что цели содержат токены-заполнители -100

In [40]:
print(targets[0])

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,   198,   198, 21017, 18261,    25,   198,
          464,  5156,   318,   355, 13779,   355,   257,  4936,    13, 50256,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100])
